# combine master files

## 필요한 모듈

이 프로젝트를 위해서는 아래의 모듈이 필요하다. 

> numpy, pandas, matplotlib, astropy, version_information

### 모듈 설치

1. 콘솔 창에서 모듈을 설치할 때는 아래와 같은 형식으로 입력하면 된다.

>pip install module_name==version

>conda install module_name==version

2. 주피터 노트북(코랩 포함)에 설치 할 때는 아래의 셀을 실행해서 실행되지 않은 모듈을 설치할 수 있다. (pip 기준) 만약 아나콘다 환경을 사용한다면 7행을 콘다 설치 명령어에 맞게 수정하면 된다.

In [1]:
# Install a pip package in the current Jupyter kernel
import importlib, sys, subprocess
print(f"sys.executable: {sys.executable}")
#!{sys.executable} -m pip install numpy==1.22 pillow==8.4 matplotlib==3.2

packages = "numpy, pandas, matplotlib, scipy, astropy, photutils, ccdproc, version_information" # required modules
pkgs = packages.split(", ")
for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f"**** {pkg} module is now installed.")
    else: 
        print(f"******** {pkg} module is already installed.")
%load_ext version_information
import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")
print(f"This notebook was generated at {now} ")

vv = %version_information {packages}
for i, pkg in enumerate(vv.packages):
    print(f"{i} {pkg[0]:10s} {pkg[1]:s}")


sys.executable: /home/guitar79/anaconda3/envs/astro_Python_env/bin/python
******** numpy module is already installed.
******** pandas module is already installed.
******** matplotlib module is already installed.
******** scipy module is already installed.
******** astropy module is already installed.
******** photutils module is already installed.
******** ccdproc module is already installed.
******** version_information module is already installed.
This notebook was generated at 2024-07-03 19:23:14 (KST = GMT+0900) 
0 Python     3.11.5 64bit [GCC 11.2.0]
1 IPython    8.25.0
2 OS         Linux 5.15.0 107 generic x86_64 with glibc2.31
3 numpy      1.26.4
4 pandas     2.2.2
5 matplotlib 3.8.4
6 scipy      1.12.0
7 astropy    6.1.0
8 photutils  1.9.0
9 ccdproc    2.4.2
10 version_information 1.0.4


### 모듈 버전 확인

아래 셀을 실행하면 이 노트북을 실행한 파이썬 및 관련 모듈의 버전을 확인할 수 있다.

### import modules

In [2]:
#%%
from glob import glob
from pathlib import Path
import os
import shutil
import numpy as np
import astropy.units as u
from astropy.stats import sigma_clip
from ccdproc import combine, ccd_process, CCDData

import ysfitsutilpy as yfu
import ysfitsutilpy as yfu
import ysphotutilpy as ypu
#import ysvisutilpy as yvu

import _astro_utilities
import _Python_utilities

In [3]:
#%%
#######################################################
BASEDIR = Path("/mnt/Rdata/OBS_data") 
PROJECDIR = Path("/mnt/Rdata/OBS_data/2024-EXO")
TODODIR = PROJECDIR / "_-_-_2024-05_-_GSON300_STF-8300M_-_1bin"
TODODIR = PROJECDIR / "_-_-_2024-06_-_GSON300_STF-8300M_-_1bin"

PROJECDIR = Path("/mnt/Rdata/OBS_data/2022-Asteroid")
TODODIR = PROJECDIR / "GSON300_STF-8300M_-_1bin"
TODODIR = PROJECDIR / "RiLA600_STX-16803_-_1bin"
TODODIR = PROJECDIR / "RiLA600_STX-16803_-_2bin"

# PROJECDIR = Path("/mnt/Rdata/OBS_data/2023-Asteroid")
# TODODIR = PROJECDIR / "GSON300_STF-8300M_-_1bin"
# TODODIR = PROJECDIR / "RiLA600_STX-16803_-_1bin"
# TODODIR = PROJECDIR / "RiLA600_STX-16803_-_2bin"

PROJECDIR = Path("/mnt/Rdata/OBS_data/2016-Variable")
TODODIR = PROJECDIR / "-_-_-_2016-_-_RiLA600_STX-16803_-_2bin"

# PROJECDIR = Path("/mnt/Rdata/OBS_data/2017-Variable")
# TODODIR = PROJECDIR / "-_-_-_2017-_-_RiLA600_STX-16803_-_2bin"

DOINGDIRs = sorted(_Python_utilities.getFullnameListOfsubDirs(TODODIR))
print ("DOINGDIRs: ", format(DOINGDIRs))
print ("len(DOINGDIRs): ", format(len(DOINGDIRs)))

MASTERDIR = [x for x in DOINGDIRs if "CAL-BDF" in str(x)]
MASTERDIR = Path(MASTERDIR[0]) / _astro_utilities.master_dir
print ("MASTERDIR: ", format(MASTERDIR))

DOINGDIRs = sorted([x for x in DOINGDIRs if "_LIGHT_" in str(x)])
print ("DOINGDIRs: ", format(DOINGDIRs))
print ("len(DOINGDIRs): ", format(len(DOINGDIRs)))

# filter_str = '2023-12-'
# DOINGDIRs = [x for x in DOINGDIRs if filter_str in x]
# remove = 'BIAS'
# DOINGDIRs = [x for x in DOINGDIRs if remove not in x]
# remove = 'DARK'
# DOINGDIRs = [x for x in DOINGDIRs if remove not in x]
# remove = 'FLAT'
# DOINGDIRs = [x for x in DOINGDIRs if remove not in x]
print ("DOINGDIRs: ", DOINGDIRs)
print ("len(DOINGDIRs): ", len(DOINGDIRs))
#######################################################

DOINGDIRs:  ['/mnt/Rdata/OBS_data/2016-Variable/-_-_-_2016-_-_RiLA600_STX-16803_-_2bin/CAL-BDF_-_-_2016-09-05_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/OBS_data/2016-Variable/-_-_-_2016-_-_RiLA600_STX-16803_-_2bin/CM-CYG_LIGHT_-_2016-10-14_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/OBS_data/2016-Variable/-_-_-_2016-_-_RiLA600_STX-16803_-_2bin/CM-CYG_LIGHT_-_2016-10-18_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/OBS_data/2016-Variable/-_-_-_2016-_-_RiLA600_STX-16803_-_2bin/CY-AQR_LIGHT_-_2016-09-05_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/OBS_data/2016-Variable/-_-_-_2016-_-_RiLA600_STX-16803_-_2bin/CY-AQR_LIGHT_-_2016-09-12_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/OBS_data/2016-Variable/-_-_-_2016-_-_RiLA600_STX-16803_-_2bin/CY-AQR_LIGHT_-_2016-09-22_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/OBS_data/2016-Variable/-_-_-_2016-_-_RiLA600_STX-16803_-_2bin/CY-AQR_LIGHT_-_2016-10-03_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/OBS_data/2016-Variable/-_-_-_2016-_-_RiLA600_STX-16803_-_2bin/CY-AQR

In [4]:
for DOINGDIR in DOINGDIRs[:1] :
    DOINGDIR = Path(DOINGDIR)
    print(f"Starting: {str(DOINGDIR.parts[-1])}")
    
    sMASTERDIR = DOINGDIR / _astro_utilities.master_dir
    REDUCEDDIR = DOINGDIR / _astro_utilities.reduced_dir

    if not sMASTERDIR.exists():
        os.makedirs(str(sMASTERDIR))
        print("{} is created...".format(str(sMASTERDIR)))

    if not REDUCEDDIR.exists():
        os.makedirs(str(REDUCEDDIR))
        print("{} is created...".format(str(REDUCEDDIR)))

    summary = yfu.make_summary(DOINGDIR/"*.fit*")
    if summary is not None :
        #print(summary)
        print("len(summary):", len(summary))
        print("summary:", summary)
        #print(summary["file"][0])

        df_light = summary.loc[summary["IMAGETYP"] == "LIGHT"].copy()
        df_light = df_light.reset_index(drop=True)

        for _, row in df_light.iterrows():

            fpath = Path(row["file"])
            ccd = yfu.load_ccd(fpath)
            filt = ccd.header["FILTER"]
            expt = ccd.header["EXPTIME"]
            # if (not (REDUCEDDIR/fpath.name).exists()) or OWrite==True :
            if (not (REDUCEDDIR/fpath.name).exists()) :
                red = yfu.ccdred(
                    ccd,
                    output=Path(f"{REDUCEDDIR/ fpath.name}"),
                    mdarkpath=str(MASTERDIR / "master_dark_{:.0f}sec.fits".format(expt)),
                    mflatpath=str(MASTERDIR / "master_flat_{}_norm.fits".format(filt.upper())),
                    # flat_norm_value=1,  # 1 = skip normalization, None = normalize by mean
                    # overwrite=OWrite,
                    )
                print (f"{REDUCEDDIR/fpath.name} is created...")

Starting: CM-CYG_LIGHT_-_2016-10-14_-_RiLA600_STX-16803_-_2bin
All 50 keywords (guessed from /mnt/Rdata/OBS_data/2016-Variable/-_-_-_2016-_-_RiLA600_STX-16803_-_2bin/CM-CYG_LIGHT_-_2016-10-14_-_RiLA600_STX-16803_-_2bin/CM-CYG_LIGHT_V_2016-10-14-12-42-17_30sec_RiLA600_STX-16803_-19c_2bin.fit) will be loaded.


DELMAG   =   0.0000            /Veriable - Comparison,   Delmag                  [astropy.io.fits.card]
DELMAG   =  -0.5755            /Veriable - Comparison,   Delmag                  [astropy.io.fits.card]
DELMAG   =  -0.6398            /Veriable - Comparison,   Delmag                  [astropy.io.fits.card]
DELMAG   =  -0.6310            /Veriable - Comparison,   Delmag                  [astropy.io.fits.card]
DELMAG   =  -0.6328            /Veriable - Comparison,   Delmag                  [astropy.io.fits.card]
DELMAG   =  -0.6036            /Veriable - Comparison,   Delmag                  [astropy.io.fits.card]
DELMAG   =  -0.6340            /Veriable - Comparison,   Delmag                  [astropy.io.fits.card]
DELMAG   =  -0.6307            /Veriable - Comparison,   Delmag                  [astropy.io.fits.card]
DELMAG   =  -0.6727            /Veriable - Comparison,   Delmag                  [astropy.io.fits.card]
DELMAG   =  -0.6428            /Veriable - Comparison,   Delmag 

len(summary): 64
summary:                                                  file  filesize  SIMPLE  \
0   /mnt/Rdata/OBS_data/2016-Variable/-_-_-_2016-_...   8395200    True   
1   /mnt/Rdata/OBS_data/2016-Variable/-_-_-_2016-_...   8395200    True   
2   /mnt/Rdata/OBS_data/2016-Variable/-_-_-_2016-_...   8395200    True   
3   /mnt/Rdata/OBS_data/2016-Variable/-_-_-_2016-_...   8395200    True   
4   /mnt/Rdata/OBS_data/2016-Variable/-_-_-_2016-_...   8395200    True   
..                                                ...       ...     ...   
59  /mnt/Rdata/OBS_data/2016-Variable/-_-_-_2016-_...   8395200    True   
60  /mnt/Rdata/OBS_data/2016-Variable/-_-_-_2016-_...   8395200    True   
61  /mnt/Rdata/OBS_data/2016-Variable/-_-_-_2016-_...   8395200    True   
62  /mnt/Rdata/OBS_data/2016-Variable/-_-_-_2016-_...   8395200    True   
63  /mnt/Rdata/OBS_data/2016-Variable/-_-_-_2016-_...   8395200    True   

    BITPIX  NAXIS  NAXIS1  NAXIS2  BSCALE    BZERO OBSERVAT  ... TELESCOP

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/Rdata/OBS_data/2016-Variable/-_-_-_2016-_-_RiLA600_STX-16803_-_2bin/CAL-BDF_-_-_2016-09-05_-_RiLA600_STX-16803_-_2bin/master_files_ys/master_dark_80sec.fits'